# MotionBERT encoder notes

MotionBERT uses `DSTformer` as its default encoder/backbone. In `thirdparty/MotionBERT/lib/utils/learning.py`, `load_backbone(args)` defaults to `args.backbone = "DSTformer"` and instantiates:

```python
DSTformer(
    dim_in=3,
    dim_out=3,
    dim_feat=args.dim_feat,
    dim_rep=args.dim_rep,
    depth=args.depth,
    num_heads=args.num_heads,
    mlp_ratio=args.mlp_ratio,
    maxlen=args.maxlen,
    num_joints=args.num_joints,
)
```

The expected input shape for the pose backbone is:

```text
[B, F, J, C]
```

For the standard MotionBERT configs this is usually:

```text
[B, 243, 17, 3]
```

- `B`: batch size
- `F`: number of frames, typically `243`
- `J`: number of joints, typically `17`
- `C`: input channels, `3`; usually x/y 2D keypoint coordinates plus confidence

`DSTformer.get_representation(x)` returns the latent representation before the final 3D pose regression head. With the default `dim_rep=512`, the latent shape is:

```text
[B, F, J, dim_rep] = [B, 243, 17, 512]
```

Relevant files:

- `thirdparty/MotionBERT/lib/model/DSTformer.py`
- `thirdparty/MotionBERT/lib/utils/learning.py`
- `thirdparty/MotionBERT/configs/pretrain/MB_pretrain.yaml`


In [ ]:
from pathlib import Path
import sys
import torch

motionbert_root = Path("thirdparty/MotionBERT").resolve()
sys.path.insert(0, str(motionbert_root))

from lib.utils.tools import get_config
from lib.utils.learning import load_backbone, load_pretrained_weights

device = "cuda" if torch.cuda.is_available() else "cpu"

config_path = motionbert_root / "configs/pretrain/MB_pretrain.yaml"
checkpoint_path = None  # Example: motionbert_root / "checkpoint/pretrain/latest_epoch.bin"

args = get_config(str(config_path))
model = load_backbone(args).to(device).eval()

if checkpoint_path is not None:
    checkpoint = torch.load(checkpoint_path, map_location=device)
    state_dict = checkpoint["model_pos"] if "model_pos" in checkpoint else checkpoint
    model = load_pretrained_weights(model, state_dict).to(device).eval()

# MotionBERT pose input: [batch, frames, joints, channels]
# Default config: [B, 243, 17, 3]
x = torch.randn(2, args.maxlen, args.num_joints, 3, device=device)

with torch.no_grad():
    latents = model.get_representation(x)
    # Equivalent:
    # latents = model(x, return_rep=True)

print("input shape:", tuple(x.shape))
print("latent shape:", tuple(latents.shape))


In [ ]:
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f"Total parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")
